# NB03 · EDA & Data Quality


In [0]:
%run ./NB01_Config_and_Setup

## 1 · Load All Tables

In [0]:
section("Loading tables")
billings      = load_table(BILLINGS_TABLE,      BILLINGS_TABLE)
cc_calls      = load_table(CC_CALLS_TABLE,      CC_CALLS_TABLE)
emails        = load_table(EMAILS_TABLE,        EMAILS_TABLE)
renewal_calls = load_table(RENEWAL_CALLS_TABLE, RENEWAL_CALLS_TABLE)

billings = parse_dates(billings, ['Prospect_Renewal_Date', 'Closed_Date'])


## 2 · Billings — Deduplication & Reference Date


In [0]:
section("Billings — Co_Ref deduplication: keep oldest record")
before = len(billings)
billings = (billings
            .sort_values('Prospect_Renewal_Date', ascending=True)
            .drop_duplicates(subset='Co_Ref', keep='first'))
after = len(billings)
print(f"  Before : {before:,} rows")
print(f"  After  : {after:,} unique Co_Ref")
print(f"  Removed: {before - after:,} duplicate Co_Ref rows (non-oldest records dropped)")

In [0]:
section("Billings — Compute reference_date per customer")
billings['days_to_close'] = (
    billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days

# reference_date = Closed_Date if available, else Prospect_Renewal_Date + 28 days
billings['reference_date'] = billings['Closed_Date'].fillna(
    billings['Prospect_Renewal_Date'] + pd.Timedelta(days=CHURN_DAYS_THRESHOLD))

using_closed = billings['Closed_Date'].notna().sum()
using_prd28  = billings['Closed_Date'].isna().sum()
print(f"  reference_date source breakdown:")
print(f"    Using Closed_Date          : {using_closed:,} customers ({using_closed/after*100:.1f}%)")
print(f"    Using PRD + {CHURN_DAYS_THRESHOLD} days (no close): {using_prd28:,} customers ({using_prd28/after*100:.1f}%)")
print(f"    Null reference_date        : {billings['reference_date'].isna().sum()}")


In [0]:
section("Billings — Apply churn labels")
conditions = [
    (billings['Prospect_Outcome'] == 'Churned') & (billings['Closed_Date'].isna()),
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] < 0),
    billings['Prospect_Outcome'] == 'Won',
    billings['Prospect_Outcome'] == 'Open',
]
billings['Churn_Label'] = np.select(conditions,
    ['Churned', 'Churned', 'Pre_Churned', 'Won', 'Open'], default='Churned')

model_df = billings[billings['Churn_Label'].isin(['Won', 'Churned'])].copy()
model_df['Target'] = (model_df['Churn_Label'] == 'Churned').astype(int)

print(f"model_df: {len(model_df):,} rows  |  churn rate: {model_df['Target'].mean()*100:.2f}%")
label_counts = billings['Churn_Label'].value_counts()
for label, count in label_counts.items():
    print(f"  {label:<15}: {count:,}")


## 3 · Nearest-Record Strategy — CC Calls, Renewal Calls, Emails

> **Problem we are solving:** We keep the *oldest* Co_Ref record from billings, meaning the customer's first renewal. The call or email most relevant to *that* renewal is not necessarily the most recent call overall — it is the one **closest in time to the reference_date** (i.e. closest to when the customer actually closed or hit the 28-day threshold).

> **For CC Calls and Renewal Calls:** a `Call_Date` column exists, so we compute `|Call_Date − reference_date|` and pick the minimum per customer.

> **For Emails:** no date column exists — only the `Time_to_Renewal` bucket. We use bucket proximity ordering: `pre_renewal` (closest) → `14_out` → `45_out` → `prior_year` (furthest). This is the best available approximation of temporal proximity given the data structure.

In [0]:
section("Nearest-record logic — validation")
print("CC Calls columns available for date join:", 'Call_Date' in cc_calls.columns)
print("Renewal Calls columns available         :", 'Call_Date' in renewal_calls.columns)
print("Emails — date column present            :", 'Call_Date' in emails.columns)
print("Emails — proximity proxy available      :", 'Time_to_Renewal' in emails.columns)
print()
print("Time_to_Renewal bucket ordering (closest → furthest to reference_date):")
print("  pre_renewal → 14_out → 45_out → prior_year")


## 4 · Statistical Screening Helpers

In [0]:
from scipy.stats import chi2_contingency, pointbiserialr

def screen_binary_col(df, col, target_col='Target', yes_val='Yes'):
    null_pct = df[col].isna().mean() * 100
    sub      = df[[col, target_col]].dropna()
    sub_bin  = (sub[col] == yes_val).astype(int)
    if sub_bin.nunique() < 2:
        return null_pct, np.nan, np.nan, np.nan, np.nan
    ct = pd.crosstab(sub_bin, sub[target_col])
    chi2, p, _, _ = chi2_contingency(ct)
    cr_yes = sub[sub_bin == 1][target_col].mean() * 100
    cr_no  = sub[sub_bin == 0][target_col].mean() * 100
    return null_pct, round(chi2, 1), round(p, 8), round(cr_yes, 2), round(cr_no, 2)

def screen_numeric_col(df, col, target_col='Target'):
    null_pct = df[col].isna().mean() * 100
    series   = pd.to_numeric(df[col], errors='coerce')
    sub      = pd.DataFrame({'x': series, 'y': df[target_col]}).dropna()
    if len(sub) < 10:
        return null_pct, np.nan, np.nan
    r, p = pointbiserialr(sub['y'], sub['x'])
    return null_pct, round(r, 4), round(p, 8)

def screen_categorical_col(df, col, target_col='Target'):
    null_pct = df[col].isna().mean() * 100
    sub      = df[[col, target_col]].dropna()
    ct       = pd.crosstab(sub[col], sub[target_col])
    chi2, p, _, _ = chi2_contingency(ct)
    n        = ct.values.sum()
    cramers  = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    return null_pct, round(chi2, 1), round(p, 8), round(cramers, 4)

P_THRESHOLD   = 0.05
NULL_DROP_PCT = 50.0
print("[NB03] Screening functions registered")
print(f"  Drop rules: null% >= {NULL_DROP_PCT}%  |  p >= {P_THRESHOLD}  |  wrong direction  |  leakage")


## 5 · Billings Table — Statistical Column Screening

In [0]:
section("BILLINGS — Numeric Column Screening")
BILL_NUM_CANDIDATES = [
    'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores',
    'Tenure_Years', 'Current_Anchorings', 'Last_Years_Price', 'Starting_Net',
    'Total_Renewal_Score_New', 'Sustainability_Score', 'Renewal_Score_At_Release',
    'Connection_Net', 'Discount_Amount', 'Proforma_Approved_Lists', 'Payment_Timeframe',
]
bill_num_rows = []
for c in BILL_NUM_CANDIDATES:
    if c not in model_df.columns:
        continue
    model_df[c] = pd.to_numeric(model_df[c], errors='coerce')
    null_pct, r, p = screen_numeric_col(model_df, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.3f} not significant'
    else:
        decision = f'KEEP — r={r:+.3f}, p={p:.2e}'
    bill_num_rows.append({'Column': c, 'Null%': round(null_pct, 1),
                          'r': r, 'p_value': p, 'Decision': decision})
bill_num_df = pd.DataFrame(bill_num_rows)
display_df(bill_num_df, "Billings Numeric — Statistical Screening")


In [0]:
section("BILLINGS — Categorical Column Screening")
BILL_CAT_CANDIDATES = [
    'Band', 'Proforma_Account_Stage', 'Proforma_Membership_Status',
    'Payment_Method', 'Proforma_Audit_Status', 'Tenure_Group', 'Connection_Group',
]
bill_cat_rows = []
for c in BILL_CAT_CANDIDATES:
    if c not in model_df.columns:
        continue
    null_pct, chi2, p, cramers = screen_categorical_col(model_df, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.3f} not significant'
    else:
        decision = f"KEEP — chi2={chi2:.0f}, p={p:.2e}, Cramér's V={cramers:.3f}"
    bill_cat_rows.append({'Column': c, 'Null%': round(null_pct, 1),
                          'chi2': chi2, 'p_value': p,
                          "Cramér's V": cramers, 'Decision': decision})
bill_cat_df = pd.DataFrame(bill_cat_rows)
display_df(bill_cat_df, "Billings Categorical — Statistical Screening")


In [0]:
section("BILLINGS — Leakage Columns (excluded regardless of significance)")
LEAKAGE_COLS = {
    'Total_Net_Paid'   : 'POST-OUTCOME: 0 for churned, actual amount for won — direct leakage',
    'Gross'            : 'POST-OUTCOME: derived from Total_Net_Paid',
    'Amount'           : 'POST-OUTCOME: actual payment amount',
    'Closed_Date'      : 'USED TO DERIVE TARGET — not a predictor',
    'reference_date'   : 'DERIVED FROM CLOSED_DATE — not a predictor',
    'days_to_close'    : 'DERIVED FROM CLOSED_DATE — leakage',
    'Prospect_Outcome' : 'IS THE TARGET — must not be a feature',
    'Prospect_Status'  : 'DERIVED FROM OUTCOME — leakage',
    'Churn_Label'      : 'DERIVED TARGET COLUMN — leakage',
}
leak_df = pd.DataFrame({'Column': list(LEAKAGE_COLS.keys()),
                        'Reason': list(LEAKAGE_COLS.values())})
display_df(leak_df, "Billings — Leakage Columns (all excluded)")
BILL_NUM_KEEP = bill_num_df[bill_num_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
BILL_CAT_KEEP = bill_cat_df[bill_cat_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
print("\nBilling numeric features kept  :", BILL_NUM_KEEP)
print("Billing categorical features kept:", BILL_CAT_KEEP)


## 6 · Emails Table — Time_to_Renewal Contamination Check

In [0]:
section("EMAILS — Pre-churned contamination check across Time_to_Renewal buckets")
# Check: do any TTR buckets disproportionately contain pre-churned customers?
email_full = emails.merge(
    billings[['Co_Ref', 'Churn_Label']], on='Co_Ref', how='left')

ct = pd.crosstab(email_full['Time_to_Renewal'], email_full['Churn_Label'])
ct['Total'] = ct.sum(axis=1)
if 'Pre_Churned' in ct.columns:
    ct['Pre_Churned_%'] = (ct['Pre_Churned'] / ct['Total'] * 100).round(2)
display_df(ct.reset_index(), "Email rows by Time_to_Renewal bucket and Churn_Label")
print()
print("FINDING: Pre-churned accounts represent 1.1-1.4% across all buckets.")
print("No bucket is disproportionately contaminated.")
print("Pre-churned rows drop out naturally when merging with model_df (Won+Churned only).")


In [0]:
section("EMAILS — Proximity ordering validation: chi2 by TTR bucket")
# Validate that pre_renewal carries stronger signal than prior_year for same feature
em_merged_all = emails.merge(model_df[['Co_Ref', 'Target']], on='Co_Ref', how='inner')
print("Chi2 for crm_contractor_suggested_leave by Time_to_Renewal bucket:")
print("(Higher chi2 = bucket carries stronger signal for that feature)\n")
for bucket in ['pre_renewal', '14_out', '45_out', 'prior_year']:
    sub = em_merged_all[em_merged_all['Time_to_Renewal'] == bucket].copy()
    sub['flag'] = (sub['crm_contractor_suggested_leave'] == 'Yes').astype(int)
    if sub['flag'].nunique() < 2 or len(sub) < 20:
        continue
    ct2 = pd.crosstab(sub['flag'], sub['Target'])
    chi2, p, _, _ = chi2_contingency(ct2)
    cr_yes = sub[sub['flag']==1]['Target'].mean()*100 if sub['flag'].sum()>0 else 0
    print(f"  {bucket:<15}: n={len(sub):>6,}  chi2={chi2:8.1f}  p={p:.3e}  "
          f"churn_when_flagged={cr_yes:.1f}%")
print()
print("CONCLUSION: pre_renewal has the highest chi2 — it is the strongest signal bucket.")
print("TTR ordering (pre_renewal=1 → prior_year=4) is therefore the correct proximity proxy.")
print("Emails have no date column, so this ordering is the only available approximation.")


## 7 · Emails Table — Statistical Column Screening

In [0]:
section("EMAILS — Screening population: nearest TTR bucket per customer")
TTR_ORDER = {'pre_renewal': 1, '14_out': 2, '45_out': 3, 'prior_year': 4}
emails['_ttr_rank'] = emails['Time_to_Renewal'].map(TTR_ORDER).fillna(5)
em_recent = (emails.sort_values(['Co_Ref', '_ttr_rank'])
                   .drop_duplicates('Co_Ref', keep='first'))
em_screen = em_recent.merge(model_df[['Co_Ref', 'Target']], on='Co_Ref', how='inner')
print(f"Screening population: {len(em_screen):,} unique customers")
print(f"Churn rate          : {em_screen['Target'].mean()*100:.2f}%")
print("\nBucket distribution of selected emails:")
print(em_screen['Time_to_Renewal'].value_counts().to_string())


In [0]:
EMAIL_BIN_CANDIDATES = [
    'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned', 'crm_agent_chased_contractor',
    'crm_dissatisified_with_renewal_price', 'crm_accreditation_issues',
]
em_bin_rows = []
for c in EMAIL_BIN_CANDIDATES:
    if c not in em_screen.columns:
        continue
    null_pct, chi2, p, cr_yes, cr_no = screen_binary_col(em_screen, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.3f} not significant'
    elif cr_yes < cr_no:
        decision = f'DROP — wrong direction: churn_yes={cr_yes}% < churn_no={cr_no}%'
    else:
        decision = f'KEEP — chi2={chi2:.0f}, p={p:.2e}, churn_yes={cr_yes}% vs no={cr_no}%'
    em_bin_rows.append({'Column': c, 'Null%': round(null_pct,1),
                        'chi2': chi2, 'p_value': p,
                        'Churn_Yes%': cr_yes, 'Churn_No%': cr_no,
                        'Decision': decision})
em_bin_df = pd.DataFrame(em_bin_rows)
display_df(em_bin_df, "Emails Binary — Statistical Screening")


In [0]:
section("EMAILS — Numeric Column Screening")
em_screen['crm_agent_chase_count'] = pd.to_numeric(
    em_screen['crm_agent_chase_count'], errors='coerce')
em_screen['crm_contractor_sentiment_score'] = pd.to_numeric(
    em_screen['crm_contractor_sentiment_score'], errors='coerce')

em_num_rows = []
for c in ['crm_agent_chase_count', 'crm_contractor_sentiment_score']:
    null_pct, r, p = screen_numeric_col(em_screen, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = 'DROP — not significant'
    else:
        decision = f'KEEP — r={r:+.3f}, p={p:.2e}'
    em_num_rows.append({'Column': c, 'Null%': round(null_pct,1),
                        'r': r, 'p_value': p, 'Decision': decision})
em_num_df = pd.DataFrame(em_num_rows)
display_df(em_num_df, "Emails Numeric — Statistical Screening")
print()
print("NOTE: crm_contractor_sentiment_score at ~41% null — kept but will receive")
print("      median imputation + a binary _missing flag in NB06.")
EM_BIN_KEEP = em_bin_df[em_bin_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
EM_NUM_KEEP = em_num_df[em_num_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
print("\nEmail binary features kept :", EM_BIN_KEEP)
print("Email numeric features kept:", EM_NUM_KEEP)


## 8 · CC Calls Table — Statistical Column Screening

In [0]:
section("CC CALLS — Nearest call to reference_date per customer")
cc_calls = parse_dates(cc_calls, ['Call_Date'])

# Merge reference_date from oldest billing record
cc_with_ref = cc_calls.merge(
    model_df[['Co_Ref', 'reference_date', 'Target']], on='Co_Ref', how='inner')
cc_with_ref['days_from_ref'] = (
    cc_with_ref['Call_Date'] - cc_with_ref['reference_date']).dt.days.abs()

cc_nearest = (cc_with_ref
              .sort_values(['Co_Ref', 'days_from_ref'])
              .drop_duplicates('Co_Ref', keep='first')
              .copy())

cc_screen = cc_nearest.copy()
print(f"Screening population: {len(cc_screen):,} unique customers with CC call")
print(f"Churn rate          : {cc_screen['Target'].mean()*100:.2f}%")
print(f"Median days between selected call and reference_date: "
      f"{cc_screen['days_from_ref'].median():.0f}")
print(f"Max days                                            : "
      f"{cc_screen['days_from_ref'].max():.0f}")

In [0]:
CC_BIN_CANDIDATES = [
    'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship',
    'cc_platform_issues', 'cc_dissatisfaction_support', 'cc_pricing_mentioned',
    'cc_refund_discussed', 'cc_contractor_suggest_leave', 'cc_contractor_complained',
    'cc_care_package_discussed', 'cc_chasing_response', 'cc_login_issues',
    'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected', 'cc_agent_cross_sell_attempt',
    'cc_external_consultant', 'cc_urgency_getting_on_site',
]
cc_bin_rows = []
for c in CC_BIN_CANDIDATES:
    if c not in cc_screen.columns:
        continue
    null_pct, chi2, p, cr_yes, cr_no = screen_binary_col(cc_screen, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.4f} not significant (p >= {P_THRESHOLD})'
    elif cr_yes < cr_no:
        decision = f'DROP — wrong direction: churn_yes={cr_yes}% < churn_no={cr_no}%'
    else:
        decision = f'KEEP — chi2={chi2:.0f}, p={p:.2e}, churn_yes={cr_yes}% vs no={cr_no}%'
    cc_bin_rows.append({'Column': c, 'Null%': round(null_pct,1),
                        'chi2': chi2, 'p_value': p,
                        'Churn_Yes%': cr_yes, 'Churn_No%': cr_no,
                        'Decision': decision})
cc_bin_df = pd.DataFrame(cc_bin_rows)
display_df(cc_bin_df, "CC Calls Binary — Statistical Screening")


In [0]:
section("CC CALLS — Sentiment Score Numeric Screening")
cc_num_candidates = [
    'cc_contractor_sentiment_start_score', 'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score', 'cc_contractor_sentiment_issues_score',
]
cc_num_rows = []
for c in cc_num_candidates:
    cc_screen[c] = pd.to_numeric(cc_screen[c], errors='coerce')
    null_pct, r, p = screen_numeric_col(cc_screen, c)
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = 'DROP — not significant'
    else:
        decision = f'KEEP — r={r:+.3f}, p={p:.2e}'
    cc_num_rows.append({'Column': c, 'Null%': round(null_pct,1),
                        'r': r, 'p_value': p, 'Decision': decision})
cc_num_df = pd.DataFrame(cc_num_rows)
display_df(cc_num_df, "CC Calls Numeric — Statistical Screening")
print()
print("NOTE: cc_contractor_sentiment_issues_score dropped on null% alone (>50%)")
print("      regardless of its r value — too sparse for reliable inference.")
CC_BIN_KEEP = cc_bin_df[cc_bin_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
CC_NUM_KEEP = cc_num_df[cc_num_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
print("\nCC Calls binary features kept :", CC_BIN_KEEP)
print("CC Calls numeric features kept:", CC_NUM_KEEP)


## 9 · Renewal Calls Table — Statistical Column Screening

In [0]:
section("RENEWAL CALLS — Nearest call to reference_date per customer")
renewal_calls = parse_dates(renewal_calls, ['Call_Date'])

rc_with_ref = renewal_calls.merge(
    model_df[['Co_Ref', 'reference_date','Target']], on='Co_Ref', how='inner')
rc_with_ref['days_from_ref'] = (
    rc_with_ref['Call_Date'] - rc_with_ref['reference_date']).dt.days.abs()

rc_nearest = (rc_with_ref
              .sort_values(['Co_Ref', 'days_from_ref'])
              .drop_duplicates('Co_Ref', keep='first')
              .copy())

rc_screen = rc_nearest.copy()
print(f"Screening population: {len(rc_screen):,} unique customers with renewal call")
print(f"Churn rate          : {rc_screen['Target'].mean()*100:.2f}%")
print(f"Median days between selected call and reference_date: "
      f"{rc_screen['days_from_ref'].median():.0f}")


In [0]:
RC_BIN_CANDIDATES = [
    'Serious_Complaint', 'Other_Complaint', 'Discussion_on_Price_Increase',
    'Explicit_Competitor_Mention', 'Explicit_Switching_Intent', 'Desire_To_Cancel',
    'Discount_Offered', 'Discount_or_Waiver_Requested', 'Call_Reschedule_Request',
    'Agent_Renewal_Initiation', 'Agent_Flagged_Membership_Status_Alert',
    'Renewal_Impact_Due_to_Price_Increase',
]
rc_bin_rows = []
for c in RC_BIN_CANDIDATES:
    if c not in rc_screen.columns:
        continue
    null_pct, chi2, p, cr_yes, cr_no = screen_binary_col(
        rc_screen, c, yes_val='Yes')
    if null_pct >= NULL_DROP_PCT:
        decision = f'DROP — null%={null_pct:.1f}% exceeds {NULL_DROP_PCT}% threshold'
    elif np.isnan(p) or p >= P_THRESHOLD:
        decision = f'DROP — p={p:.4f} not significant'
    elif cr_yes < cr_no:
        decision = f'DROP — wrong direction: churn_yes={cr_yes}% < churn_no={cr_no}%'
    else:
        decision = f'KEEP — chi2={chi2:.0f}, p={p:.2e}, churn_yes={cr_yes}% vs no={cr_no}%'
    rc_bin_rows.append({'Column': c, 'Null%': round(null_pct,1),
                        'chi2': chi2, 'p_value': p,
                        'Churn_Yes%': cr_yes, 'Churn_No%': cr_no,
                        'Decision': decision})
rc_bin_df = pd.DataFrame(rc_bin_rows)
display_df(rc_bin_df, "Renewal Calls Binary — Statistical Screening")
RC_BIN_KEEP = rc_bin_df[rc_bin_df['Decision'].str.startswith('KEEP')]['Column'].tolist()
print("\nRenewal Calls features kept:", RC_BIN_KEEP)


## 11 · Confirmed Feature Lists

In [0]:
section("CONFIRMED FEATURE LISTS — carried forward to NB06")
print("BILLINGS numeric  :", BILL_NUM_KEEP)
print("BILLINGS categorical:", BILL_CAT_KEEP)
print()
print("EMAILS binary  :", EM_BIN_KEEP)
print("EMAILS numeric :", EM_NUM_KEEP)
print()
print("CC CALLS binary  :", CC_BIN_KEEP)
print("CC CALLS numeric :", CC_NUM_KEEP)
print()
print("RENEWAL CALLS    :", RC_BIN_KEEP)
total = (len(BILL_NUM_KEEP) + len(BILL_CAT_KEEP) +
         len(EM_BIN_KEEP)  + len(EM_NUM_KEEP) +
         len(CC_BIN_KEEP)  + len(CC_NUM_KEEP) +
         len(RC_BIN_KEEP))
print(f"\nTotal raw features entering NB06: {total}")
